# Lexel Stage 2: Masked-SimPO / DPO Alignment, Held-Out Evaluation Gate & GGUF Export

This notebook runs Stage 2 Preference Alignment on surgical counterfactual pairs (`lexel_dpo_train.jsonl`), verifies the aligned model against the 13-surface held-out benchmark (`lexel_heldout_eval.jsonl`), and exports quantized GGUF weights matching Lexel Desktop's `models/manifest.json`.

## Pipeline Stages
1. **Load Stage 1 SFT Adapter**: Resumes from `./outputs/<tier>-sft-adapter/adapter`.
2. **Surgical Token-Masked Alignment**: Aligns on `lexel_dpo_train.jsonl` using reference-free `MaskedSimPOLoss` (`beta=2.0`, `gamma=1.0`, `sft_weight=0.05`, `unmasked_token_weight=0.05`, `lr=5e-6`, `1.5 epochs`) or Unsloth `PatchDPOTrainer`.
3. **Deterministic Held-Out Evaluation Gate**: Evaluates all 13 surfaces in `lexel_heldout_eval.jsonl` and requires:
   - Zero-Slop & Zero-Em-Dash Rate >= 95%
   - Unit-Bound Number & Code/URL/CLI Token Preservation == 100%
   - Surface Compliance >= 90%
   - Exact 95% Clopper-Pearson confidence intervals.
4. **GGUF Export & Manifest Generation**: Exports `q4_k_m` (plus optional `q5_k_m`, `q8_0`) GGUF files (`qwen2.5-1.5b-instruct-q4_k_m.gguf` or `qwen2.5-0.5b-instruct-q4_k_m.gguf`), writes `SHA256SUMS`, and emits `manifest.json` matching `models/manifest.json`.

In [ ]:
# 1. Imports & Environment Setup
import hashlib
import json
import os
import sys
import time
from datetime import UTC, datetime
from pathlib import Path

import torch
from datasets import load_dataset
from trl import DPOConfig, DPOTrainer
from unsloth import FastLanguageModel, PatchDPOTrainer, is_bfloat16_supported

PatchDPOTrainer()

REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ml.crucible.evaluate import (
    WritingBenchCritic,
    clopper_pearson_interval,
    evaluate_heldout_generations,
    sign_test_p_value,
)
from ml.crucible.loss import MaskedSimPOLoss
from ml.crucible.verifier import sanitize_model_output, verify_rewrite

In [ ]:
# 2. Configuration & Stage 1 Adapter Loading
MODEL_TIER = os.environ.get("LEXEL_MODEL_TIER", "lexel-1.5b")
MAX_SEQ_LENGTH = 1536

DATA_DIR = Path(os.environ.get("LEXEL_DATA_DIR", "../datasets"))
DPO_FILE = DATA_DIR / "lexel_dpo_train.jsonl"
HELDOUT_FILE = DATA_DIR / "lexel_heldout_eval.jsonl"

STAGE1_ADAPTER_DIR = Path(
    os.environ.get("LEXEL_STAGE1_ADAPTER", f"./outputs/{MODEL_TIER}-sft-adapter/adapter")
)
STAGE2_OUT_DIR = Path(
    os.environ.get("LEXEL_STAGE2_OUT", f"./outputs/{MODEL_TIER}-dpo-aligned")
)
GGUF_OUT_DIR = STAGE2_OUT_DIR / "gguf"
STAGE2_OUT_DIR.mkdir(parents=True, exist_ok=True)
GGUF_OUT_DIR.mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(STAGE1_ADAPTER_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Configure reference-free MaskedSimPOLoss with surgical token masking
simpo_loss_fn = MaskedSimPOLoss(
    beta=2.0,
    gamma=1.0,
    sft_weight=0.05,
    unmasked_token_weight=0.05,
)
print("Loaded Stage 1 adapter and initialized MaskedSimPOLoss:", simpo_loss_fn.__dict__)

In [ ]:
# 3. Load Surgical Counterfactual Preference Dataset & Run Stage 2 Alignment
dpo_dataset = load_dataset("json", data_files={"train": str(DPO_FILE)})["train"]
print(f"Loaded {len(dpo_dataset)} surgical counterfactual preference pairs.")

dpo_config = DPOConfig(
    output_dir=str(STAGE2_OUT_DIR),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.10,
    num_train_epochs=1.5,
    beta=2.0,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=768,
    optim="adamw_8bit",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    save_strategy="epoch",
    seed=42,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
)

t0 = time.perf_counter()
dpo_stats = dpo_trainer.train()
dpo_elapsed_sec = time.perf_counter() - t0
print(f"Stage 2 preference alignment finished in {dpo_elapsed_sec:.1f}s (loss={dpo_stats.training_loss:.4f})")

In [ ]:
# 4. Held-Out Benchmark Evaluation Gate (All 13 Surfaces)
FastLanguageModel.for_inference(model)
heldout_rows = [
    json.loads(line)
    for line in HELDOUT_FILE.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

critic = WritingBenchCritic()
evaluated_records = []
pairwise_wins = 0
pairwise_decisive = 0

for row in heldout_rows:
    inputs = tokenizer.apply_chat_template(
        row["messages"],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=280,
            temperature=0.2,
            do_sample=False,
            use_cache=True,
        )
    generated_ids = outputs[0][inputs.shape[-1] :]
    raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    candidate_text = sanitize_model_output(raw_text)

    rec_copy = dict(row)
    rec_copy["candidate"] = candidate_text
    evaluated_records.append(rec_copy)

    # Compare aligned candidate vs unedited sloppy source using WritingBenchCritic
    cmp_res = critic.evaluate_pair(
        {"source_material": row["source"], "surface_id": row["surface_id"]},
        candidate_text,
        row["source"],
    )
    if cmp_res.winner != "tie":
        pairwise_decisive += 1
        if cmp_res.winner == "A":
            pairwise_wins += 1

eval_summary = evaluate_heldout_generations(evaluated_records, candidate_key="candidate")
eval_summary["pairwise_win_rate_vs_source"] = round(
    pairwise_wins / max(1, pairwise_decisive), 4
)
eval_summary["pairwise_sign_test_p"] = sign_test_p_value(pairwise_wins, pairwise_decisive)

print(json.dumps(eval_summary, indent=2))

assert eval_summary["zero_slop_rate"] >= 0.95, f"Zero-slop gate failed: {eval_summary['zero_slop_rate']}"
assert eval_summary["number_preservation_rate"] == 1.0, f"Number preservation gate failed: {eval_summary['number_preservation_rate']}"
assert eval_summary["token_preservation_rate"] == 1.0, f"Token preservation gate failed: {eval_summary['token_preservation_rate']}"
assert eval_summary["surface_compliance_rate"] >= 0.90, f"Surface compliance gate failed: {eval_summary['surface_compliance_rate']}"
print("All Held-Out Benchmark Quality Gates PASSED.")

In [ ]:
# 5. Export GGUF Weights (q4_k_m + optional q5_k_m, q8_0) & Build Lexel Desktop manifest.json
QUANT_METHODS = ["q4_k_m"]  # Add "q5_k_m" or "q8_0" when exporting additional quantizations

for q_method in QUANT_METHODS:
    model.save_pretrained_gguf(
        str(GGUF_OUT_DIR),
        tokenizer,
        quantization_method=q_method,
    )

CANONICAL_GGUF_NAMES = {
    "lexel-1.5b": "qwen2.5-1.5b-instruct-q4_k_m.gguf",
    "lexel-0.5b": "qwen2.5-0.5b-instruct-q4_k_m.gguf",
}
canonical_filename = CANONICAL_GGUF_NAMES[MODEL_TIER]
exported_ggufs = sorted(GGUF_OUT_DIR.glob("*.gguf"))
if exported_ggufs and not (GGUF_OUT_DIR / canonical_filename).exists():
    exported_ggufs[0].rename(GGUF_OUT_DIR / canonical_filename)

target_gguf = GGUF_OUT_DIR / canonical_filename
gguf_sha256 = hashlib.sha256(target_gguf.read_bytes()).hexdigest()
gguf_size_bytes = target_gguf.stat().st_size

sha256sums_path = GGUF_OUT_DIR / "SHA256SUMS"
sha256sums_path.write_text(f"{gguf_sha256}  {canonical_filename}\n", encoding="utf-8")

# Generate manifest.json matching Lexel Desktop models/manifest.json schema
desktop_manifest = {
    "version": 1,
    "defaultMainModel": "lexel-1.5b",
    "models": [
        {
            "id": "lexel-0.5b",
            "tier": "lite",
            "label": "Lexel 0.5B (Lite)",
            "description": "Fast CPU/GPU model (~469 MB) for 4-8 GB laptops and quick edits.",
            "ollamaTag": "qwen2.5:0.5b",
            "ggufRepo": "Qwen/Qwen2.5-0.5B-Instruct-GGUF",
            "ggufFilename": "qwen2.5-0.5b-instruct-q4_k_m.gguf",
            "ggufUrl": "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/qwen2.5-0.5b-instruct-q4_k_m.gguf",
            "sha256": gguf_sha256 if MODEL_TIER == "lexel-0.5b" else "74a4da8c9fdbcd15bd1f6d01d621410d31c6fc00986f5eb687824e7b93d7a9db",
            "sizeBytes": gguf_size_bytes if MODEL_TIER == "lexel-0.5b" else 491400032,
            "approxRamMb": 450,
        },
        {
            "id": "lexel-1.5b",
            "tier": "standard",
            "label": "Lexel 1.5B (Standard)",
            "description": "Primary rewrite model (~1.04 GB) balancing prose quality and local latency.",
            "ollamaTag": "qwen2.5:1.5b",
            "ggufRepo": "Qwen/Qwen2.5-1.5B-Instruct-GGUF",
            "ggufFilename": "qwen2.5-1.5b-instruct-q4_k_m.gguf",
            "ggufUrl": "https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/qwen2.5-1.5b-instruct-q4_k_m.gguf",
            "sha256": gguf_sha256 if MODEL_TIER == "lexel-1.5b" else "6a1a2eb6d15622bf3c96857206351ba97e1af16c30d7a74ee38970e434e9407e",
            "sizeBytes": gguf_size_bytes if MODEL_TIER == "lexel-1.5b" else 1117320736,
            "approxRamMb": 1050,
        },
    ],
}

(GGUF_OUT_DIR / "manifest.json").write_text(
    json.dumps(desktop_manifest, indent=2) + "\n", encoding="utf-8"
)
print(f"Exported {canonical_filename} ({gguf_size_bytes} bytes, sha256={gguf_sha256})")